In [3]:
# ============================================================
# Cell 1 — Install & Imports
# ============================================================
!pip install -q faiss-cpu datasets psutil

import numpy as np
import faiss
import gc, os, ctypes, time
import psutil
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from datasets import load_dataset

def free_ram_gb():  return psutil.virtual_memory().available / 1e9
def used_ram_gb():  return psutil.virtual_memory().used / 1e9

def aggressive_gc():
    gc.collect(); gc.collect()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except: pass

print(f"FAISS {faiss.__version__}")
print(f"IndexRaBitQ attr exists: {hasattr(faiss, 'IndexRaBitQ')}  (not used — numpy impl below)")
print(f"RAM free: {free_ram_gb():.1f} GB")

FAISS 1.13.0
IndexRaBitQ attr exists: True  (not used — numpy impl below)
RAM free: 23.3 GB


In [1]:
def load_glove_200():
    data_dir = "/scratch/ss20308/c3/pp/glove200"
    os.makedirs(data_dir, exist_ok=True)

    if not os.path.exists(f"{data_dir}/base.npy"):
        import h5py

        # Path where Colab puts uploaded files
        hdf5_path = "/scratch/ss20308/c3/pp/glove-200-angular.hdf5"
        
        if not os.path.exists(hdf5_path):
            raise FileNotFoundError(
                "Please upload glove-200-angular.hdf5 to Colab first!\n"
                "Use the file upload button in the left sidebar → Files → Upload"
            )

        print("Reading HDF5 …")
        with h5py.File(hdf5_path, "r") as f:
            print(f"  Keys: {list(f.keys())}")
            X_full  = np.array(f["train"], dtype=np.float32)
            Xq_full = np.array(f["test"],  dtype=np.float32)
        
        print(f"  Full base: {X_full.shape}  Queries: {Xq_full.shape}")

        # Normalize
        norms = np.linalg.norm(X_full, axis=1)
        print(f"  Norm mean: {norms.mean():.4f}")
        if abs(norms.mean() - 1.0) > 0.01:
            print("  Normalizing …")
            X_full  = X_full  / np.maximum(np.linalg.norm(X_full,  axis=1, keepdims=True), 1e-12)
            Xq_full = Xq_full / np.maximum(np.linalg.norm(Xq_full, axis=1, keepdims=True), 1e-12)

        # Sample 100k base
        print(f"  Sampling {N_BASE} from {len(X_full)} base vectors …")
        rng = np.random.RandomState(42)
        idx = rng.choice(len(X_full), N_BASE, replace=False)
        idx.sort()
        X = np.ascontiguousarray(X_full[idx])
        del X_full; aggressive_gc()

        # Use N_QUERY queries
        Xq = np.ascontiguousarray(Xq_full[:N_QUERY])
        del Xq_full; aggressive_gc()

        # Recompute GT with FAISS against sampled base
        print("  Recomputing GT with FAISS …")
        # In load_glove_200 — change to IndexFlatIP
        idx_flat = faiss.IndexFlatIP(200)
        idx_flat.add(X)
        _, GT = idx_flat.search(Xq, K_GT)
        GT = np.ascontiguousarray(GT.astype(np.int64))
        del idx_flat; aggressive_gc()

        np.save(f"{data_dir}/base.npy",       X)
        np.save(f"{data_dir}/query.npy",       Xq)
        np.save(f"{data_dir}/groundtruth.npy", GT)
        print("  Saved to disk.")

    else:
        print("  Loading from cache …")
        X  = np.load(f"{data_dir}/base.npy")
        Xq = np.load(f"{data_dir}/query.npy")
        GT = np.load(f"{data_dir}/groundtruth.npy")

    print(f"GloVe-200: Base {X.shape}  Queries {Xq.shape}  GT {GT.shape}")
    return X, Xq, GT

In [ ]:
import numpy as np
import faiss
import time
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# =========================
# CONFIG
# =========================
KS = [1,2,4,8,16,32]
K_MAX = max(KS)

# =========================
# METRICS
# =========================
def recall_at_k(I, GT):
    res = {}
    for k in KS:
        correct = 0
        for i in range(len(I)):
            if GT[i,0] in I[i,:k]:
                correct += 1
        res[k] = correct / len(I)
    return res

# =========================
# LOAD DATA (uses your loader)
# =========================
def load_data():
    X, Xq, _ = load_glove_200()

    rng = np.random.RandomState(42)
    X  = X[rng.choice(len(X), 100000, replace=False)]
    Xq = Xq[rng.choice(len(Xq), 10000, replace=False)]

    faiss.normalize_L2(X)
    faiss.normalize_L2(Xq)

    return np.ascontiguousarray(X), np.ascontiguousarray(Xq)

# =========================
# GROUND TRUTH
# =========================
def compute_gt(X, Xq):
    index = faiss.IndexFlatIP(X.shape[1])
    index.add(X)
    _, GT = index.search(Xq, K_MAX)
    return GT

# =========================
# PQ flat exhaustive search
# =========================
def run_pq(X, Xq, GT, bits_per_coord):
    dim = X.shape[1]
    group_size = 4 if bits_per_coord == 2 else 2
    m = dim // group_size
    nbits = bits_per_coord * group_size  # = 8

    idx = faiss.IndexPQ(dim, m, nbits, faiss.METRIC_INNER_PRODUCT)
    idx.train(X)
    idx.add(X)

    _, I = idx.search(Xq, K_MAX)
    return recall_at_k(I, GT)

# =========================
# RaBitQ (flat + sign)
# =========================
def train_rabitq(X, bits):
    dim = X.shape[1]

    print("Rotation...")
    R = np.linalg.qr(np.random.randn(dim, dim))[0].astype(np.float32)
    Xr = X @ R

    print("Sign + magnitude")
    sign = (Xr >= 0).astype(np.int8)
    mag  = np.abs(Xr)

    levels = 2 ** bits
    min_val = mag.min(axis=0)
    max_val = mag.max(axis=0)

    scale = (max_val - min_val) / levels
    scale[scale == 0] = 1e-6

    codes = np.floor((mag - min_val) / scale).astype(np.int32)

    return {"R":R,"sign":sign,"codes":codes,"min":min_val,"scale":scale}

def search_rabitq(model, Xq, batch_size=256):
    R, sign, codes = model["R"], model["sign"], model["codes"]
    min_val, scale = model["min"], model["scale"]

    Xqr = Xq @ R
    q_sign = (Xqr >= 0).astype(np.int8)
    q_mag  = np.abs(Xqr)

    # reconstruct DB once
    X_rec = codes * scale + min_val
    X_rec = X_rec * (2*sign - 1)

    all_I = []
    for i in tqdm(range(0, len(Xq), batch_size)):
        qb = q_mag[i:i+batch_size]
        sb = q_sign[i:i+batch_size]

        q_rec = qb * (2*sb - 1)

        D = q_rec @ X_rec.T
        I = np.argsort(-D, axis=1)[:, :K_MAX]
        all_I.append(I)

    return np.vstack(all_I)

# =========================
# RUN
# =========================
print("Loading...")
X, Xq = load_data()

print("GT...")
GT = compute_gt(X, Xq)

results = {}

# ---- PQ ----
print("\n=== PQ ===")
results["PQ-2bit"] = run_pq(X, Xq, GT, 2)
results["PQ-4bit"] = run_pq(X, Xq, GT, 4)

# ---- RaBitQ ----
print("\n=== RaBitQ ===")
for b in [2,4]:
    t0 = time.time()
    model = train_rabitq(X, b)
    I = search_rabitq(model, Xq)
    rec = recall_at_k(I, GT)

    results[f"RaBitQ-{b}bit+sign"] = rec

    print(f"\nRaBitQ-{b}bit+sign:",
          " ".join([f"R@{k}:{rec[k]:.4f}" for k in KS]),
          f"| bits/dim ≈ {b+1} | time {time.time()-t0:.1f}s")

# =========================
# PRINT TABLE
# =========================
print("\n=== FINAL RESULTS ===")
for name, rec in results.items():
    print(name, ":", " ".join([f"R@{k}:{rec[k]:.4f}" for k in KS]))

# =========================
# PLOT: Recall@k
# =========================
plt.figure(figsize=(7,5))
for name, rec in results.items():
    y = [rec[k] for k in KS]
    plt.plot(KS, y, marker='o', linewidth=2, label=name)

plt.xlabel("k")
plt.ylabel("Recall@k")
plt.title("Flat Comparison: PQ vs RaBitQ (+sign)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

Loading...
  Loading from cache …
GloVe-200: Base (100000, 200)  Queries (10000, 200)  GT (10000, 32)
GT...

=== PQ ===

=== RaBitQ ===
Rotation...
Sign + magnitude


  0%|          | 0/40 [00:00<?, ?it/s]